In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('data/Dry_Eye_Dataset.csv')

# tidy column names
df.columns = (df.columns.str.lower()
              .str.replace('/', '_', regex=False)
              .str.replace('-', '_', regex=False)
              .str.replace(' ', '_', regex=False))

# blood pressure "137/89" -> two numeric columns
df[['systolic', 'diastolic']] = df.blood_pressure.str.split('/', expand=True).astype(int)
del df['blood_pressure']

# Y/N columns -> 1/0 (includes the target)
yn_cols = [c for c in df.columns if set(df[c].unique()) <= {'Y', 'N'}]
for c in yn_cols:
    df[c] = (df[c] == 'Y').astype(int)

df['gender'] = df.gender.str.lower()          # f / m
print(yn_cols)
df.head()

['sleep_disorder', 'wake_up_during_night', 'feel_sleepy_during_day', 'caffeine_consumption', 'alcohol_consumption', 'smoking', 'medical_issue', 'ongoing_medication', 'smart_device_before_bed', 'blue_light_filter', 'discomfort_eye_strain', 'redness_in_eye', 'itchiness_irritation_in_eye', 'dry_eye_disease']


,gender,age,sleep_duration,sleep_quality,stress_level,heart_rate,daily_steps,physical_activity,height,weight,...,ongoing_medication,smart_device_before_bed,average_screen_time,blue_light_filter,discomfort_eye_strain,redness_in_eye,itchiness_irritation_in_eye,dry_eye_disease,systolic,diastolic
0,f,24,9.5,2,1,67,3000,31,161,69,...,1,0,8.7,0,1,1,0,1,137,89
1,m,39,9.6,2,3,60,12000,74,164,87,...,1,0,9.6,1,0,0,1,1,108,64
2,f,45,5.4,1,5,95,12000,93,179,94,...,0,1,4.0,0,1,0,0,0,134,81
3,f,45,5.4,4,5,78,19000,32,160,77,...,0,0,7.6,0,1,0,1,0,110,90
4,f,42,5.7,3,2,72,4000,173,179,99,...,0,0,3.5,0,1,1,0,1,99,67


In [2]:
print('shape:', df.shape)
print('missing:', df.isnull().sum().sum(), '| duplicate rows:', df.duplicated().sum())
print()
print('target balance:')
print(df.dry_eye_disease.value_counts(normalize=True).round(3))
print()
df.describe().round(1).T

shape: (20000, 27)
missing: 0 | duplicate rows: 0

target balance:
dry_eye_disease
1    0.652
0    0.348
Name: proportion, dtype: float64



,count,mean,std,min,25%,50%,75%,max
age,20000.0,31.4,8.1,18.0,24.0,31.0,39.0,45.0
sleep_duration,20000.0,7.0,1.7,4.0,5.5,7.0,8.5,10.0
sleep_quality,20000.0,3.0,1.4,1.0,2.0,3.0,4.0,5.0
stress_level,20000.0,3.0,1.4,1.0,2.0,3.0,4.0,5.0
heart_rate,20000.0,79.9,11.8,60.0,70.0,80.0,90.0,100.0
daily_steps,20000.0,10536.9,5752.7,1000.0,6000.0,11000.0,16000.0,20000.0
physical_activity,20000.0,90.1,52.3,0.0,45.0,91.0,135.0,180.0
height,20000.0,174.9,14.7,150.0,162.0,175.0,188.0,200.0
weight,20000.0,74.9,14.7,50.0,62.0,75.0,88.0,100.0
sleep_disorder,20000.0,0.5,0.5,0.0,0.0,0.0,1.0,1.0


In [3]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

In [4]:
df['gender_m'] = (df.gender == 'm').astype(int)

In [5]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=1)


In [6]:
target = 'dry_eye_disease'
symptoms = ['discomfort_eye_strain', 'redness_in_eye', 'itchiness_irritation_in_eye']
all_features = [c for c in df.columns if c not in ['gender', target]]
lifestyle = [c for c in all_features if c not in symptoms]

for name, cols in [('all features', all_features), ('lifestyle only', lifestyle)]:
    for model in [LogisticRegression(max_iter=2000),
                  RandomForestClassifier(n_estimators=100, max_depth=8,
                                         random_state=1, n_jobs=-1)]:
        model.fit(df_train[cols], df_train[target])
        auc = roc_auc_score(df_val[target], model.predict_proba(df_val[cols])[:, 1])
        print(f'{name:15s} {type(model).__name__:25s} AUC = {auc:.3f}')

/home/yessine/practice/dry-eye-disease-prediction/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 2000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=2000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


all features    LogisticRegression        AUC = 0.594
all features    RandomForestClassifier    AUC = 0.597


/home/yessine/practice/dry-eye-disease-prediction/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 2000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=2000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


lifestyle only  LogisticRegression        AUC = 0.490
lifestyle only  RandomForestClassifier    AUC = 0.508


In [7]:
sym = df_val[target].groupby([df_val[c] for c in symptoms]).agg(['mean', 'count']).round(3)
print(sym)

                                                                   mean  count
discomfort_eye_strain redness_in_eye itchiness_irritation_in_eye              
0                     0              0                            0.303    468
                                     1                            0.692    497
                      1              0                            0.673    511
                                     1                            0.688    522
1                     0              0                            0.680    493
                                     1                            0.722    514
                      1              0                            0.722    492
                                     1                            0.694    503


In [8]:
from sklearn.metrics import mutual_info_score

num = [c for c in all_features]
print(df_full_train[num].corrwith(df_full_train[target]).sort_values(key=abs, ascending=False).round(3).head(10))

mi = {c: mutual_info_score(df_full_train[c], df_full_train[target])
      for c in all_features if df_full_train[c].nunique() <= 6}
print(pd.Series(mi).sort_values(ascending=False).round(4).head(10))

discomfort_eye_strain          0.111
redness_in_eye                 0.109
itchiness_irritation_in_eye    0.100
gender_m                      -0.020
average_screen_time            0.016
smart_device_before_bed       -0.016
diastolic                     -0.014
alcohol_consumption            0.013
ongoing_medication            -0.008
smoking                       -0.007
dtype: float64
discomfort_eye_strain          0.0062
redness_in_eye                 0.0059
itchiness_irritation_in_eye    0.0051
gender_m                       0.0002
smart_device_before_bed        0.0001
alcohol_consumption            0.0001
sleep_quality                  0.0001
stress_level                   0.0001
ongoing_medication             0.0000
smoking                        0.0000
dtype: float64


In [9]:
from sklearn.model_selection import cross_val_score, KFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from itertools import product

cv = KFold(n_splits=5, shuffle=True, random_state=1)
y = df_full_train[target]

def cv_auc(model, cols):
    s = cross_val_score(model, df_full_train[cols], y, cv=cv, scoring='roc_auc', n_jobs=-1)
    return s.mean(), s.std()

feature_sets = {'all': all_features, 'symptoms_only': symptoms, 'lifestyle_only': lifestyle}
results = []

for fs_name, cols in feature_sets.items():
    for C in [0.01, 0.1, 1, 10]:
        m = make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=2000))
        results.append((fs_name, 'logreg', f'C={C}', *cv_auc(m, cols)))

    for d, leaf in product([2, 3, 4, 6], [10, 50, 100]):
        m = DecisionTreeClassifier(max_depth=d, min_samples_leaf=leaf, random_state=1)
        results.append((fs_name, 'tree', f'depth={d},leaf={leaf}', *cv_auc(m, cols)))

    for n, d, leaf in product([100, 200], [3, 5, 8], [10, 50]):
        m = RandomForestClassifier(n_estimators=n, max_depth=d, min_samples_leaf=leaf,
                                   random_state=1, n_jobs=1)
        results.append((fs_name, 'forest', f'n={n},depth={d},leaf={leaf}', *cv_auc(m, cols)))

    for eta, d, n in product([0.03, 0.1], [2, 3, 4], [100, 200]):
        m = XGBClassifier(learning_rate=eta, max_depth=d, n_estimators=n,
                          random_state=1, n_jobs=1, eval_metric='auc')
        results.append((fs_name, 'xgboost', f'eta={eta},depth={d},n={n}', *cv_auc(m, cols)))

df_res = pd.DataFrame(results, columns=['features', 'model', 'params', 'auc_mean', 'auc_std'])
best = df_res.sort_values('auc_mean', ascending=False).groupby(['features', 'model']).head(1)
best.sort_values(['features', 'auc_mean'], ascending=[True, False]).round(4)

,features,model,params,auc_mean,auc_std
17,all,forest,"n=100,depth=3,leaf=50",0.6008,0.0160
3,all,logreg,C=10,0.6005,0.0147
29,all,xgboost,"eta=0.03,depth=2,n=200",0.6002,0.0166
15,all,tree,"depth=6,leaf=100",0.5955,0.0178
89,lifestyle_only,tree,"depth=3,leaf=100",0.5071,0.0073
116,lifestyle_only,xgboost,"eta=0.1,depth=3,n=100",0.5027,0.0094
100,lifestyle_only,forest,"n=100,depth=8,leaf=10",0.5025,0.0139
83,lifestyle_only,logreg,C=10,0.5001,0.0104
74,symptoms_only,xgboost,"eta=0.1,depth=2,n=100",0.5988,0.0149
42,symptoms_only,logreg,C=1,0.5980,0.0156
